## Import Libraries

In [ ]:
# ── Notebook bootstrap: resolve the production module path ─────────────────
# This single cell replaces all duplicated production logic.
# Strategy: insert public/image-analysis-miu-batubara/ into sys.path so that
#   `import circle_detection` and `import block_detection` resolve to the
#   exact same source files served by the PyScript production environment.
# The path is computed from the repo root, found by walking up from this
#   notebook file.  VS Code exposes __vsc_ipynb_file__; for other kernels we
#   fall back to Path.cwd() (reliable when the server is started from the root).

import sys
from pathlib import Path

def _find_repo_root() -> Path:
    """Walk up from the notebook location until we find pytest.ini (repo marker)."""
    # VS Code Jupyter sets __vsc_ipynb_file__ to the notebook's absolute path.
    try:
        start = Path(__vsc_ipynb_file__).resolve().parent  # noqa: F821
    except NameError:
        start = Path.cwd()  # fallback: assumes kernel was started from repo root

    for candidate in [start, *start.parents]:
        if (candidate / "pytest.ini").exists():
            return candidate
    # Last resort: return start (notebook dir / CWD)
    return start

_REPO_ROOT = _find_repo_root()
_APP_DIR = str(_REPO_ROOT / "public" / "image-analysis-miu-batubara")

if _APP_DIR not in sys.path:
    sys.path.insert(0, _APP_DIR)

print(f"✅ Production module path: {_APP_DIR}")
print(f"   Resolved repo root   : {_REPO_ROOT}")


In [ ]:
# ── Production imports (Single Source of Truth) ────────────────────────────
# All detection logic lives exclusively in block_detection.py.
# No code is duplicated here.
from block_detection import (
    process_blocks,
    analyze_block_histograms,
    subdivide_blocks,
    analyze_subdivision_histograms,
    visualize_block_invalid_roi,
    compare_blocks_1_vs_3,
    AIR_STEP_MAX_REL_DIFF,
    AIR_BLOCK_VALIDATION_CODE,
    AIR_BLOCK_VALIDATION_ERROR,
    ROI_SHRINK_RATIO,
)
print("✅ block_detection imported from production source.")


## Define Image Processing Function

In [ ]:
# ── Notebook-only helpers (not in production) ──────────────────────────────
from pathlib import Path


def _to_file_bytes(file_or_bytes):
    if isinstance(file_or_bytes, (bytes, bytearray)):
        return bytes(file_or_bytes)
    if isinstance(file_or_bytes, Path):
        return file_or_bytes.read_bytes()
    if isinstance(file_or_bytes, str):
        return Path(file_or_bytes).read_bytes()
    raise TypeError(f"Unsupported input type: {type(file_or_bytes)!r}")


def process_image(image_path, **params):
    """Notebook convenience: accepts a file path as well as raw bytes."""
    return process_blocks(_to_file_bytes(image_path), params or {})


def run_full_miu_analysis(
    image_path,
    detect_params=None,
    compare_params=None,
    num_subdivisions=10,
    scale_factor=2 / 3,
):
    """End-to-end production-identical block MIU pipeline for notebook debugging."""
    file_bytes = _to_file_bytes(image_path)
    detected = process_blocks(file_bytes, detect_params or {})
    subs     = subdivide_blocks(
        file_bytes,
        detected["blocks"],
        num_subdivisions=num_subdivisions,
        scale_factor=scale_factor,
    )
    analysis = compare_blocks_1_vs_3(file_bytes, subs, params=compare_params or {})
    return {
        "detected":     detected,
        "subdivisions": subs,
        "analysis":     analysis,
    }


print("✅ Notebook helpers defined (process_image, run_full_miu_analysis).")


## Step 1 — Detect Blocks

Specify the TIFF image path and run block detection.

In [ ]:
# ── Step 1: Detect blocks ──────────────────────────────────────────────────
image_file = r"sample-block.tiff"  # ← update to your TIFF path

detect_params = {
    "threshold_value": 55000,
    "min_length_rectangular": 1400,
    "max_length_rectangular": 1600,
    "min_rectangularity": 0.9,
    "min_solidity": 0.9,
}

detection_result = process_image(image_file, **detect_params)
all_blocks = detection_result["blocks"]

print(f"Detected {detection_result['count']} blocks")
for b in all_blocks:
    print(f"  Block {b['id']} ({b['type']:10}) center={b['center']}  "
          f"mean={b['mean_value']:.1f}")


## Step 2 — Subdivide Blocks into 10 Step-Wedge Grids

Divide each block into 10 equal subdivisions along its longest side.

In [ ]:
# ── Step 2: Subdivide each block into 10 step-wedge subdivisions ───────────
subdivisions = subdivide_blocks(image_file, all_blocks, num_subdivisions=10)
print(f"Total subdivisions: {subdivisions['total_count']}")
print(f"Subdivisions per block: {subdivisions['num_subdivisions']}")


## Step 3 — Histogram Analysis

Plot pixel-value histograms for each block and each subdivision.

In [ ]:
# ── Step 3: Histogram analysis for each block ──────────────────────────────
hist_result = analyze_block_histograms(image_file, all_blocks)
if hist_result:
    print("Block histogram generated successfully.")
else:
    print("Histogram analysis returned None (check image path).")


In [ ]:
# ── Step 4: Histogram for each subdivision of Block 1 ─────────────────────
sub_hist = analyze_subdivision_histograms(image_file, subdivisions, block_number=1)
if sub_hist:
    print(f"Subdivision histogram for Block {sub_hist['block_number']} generated.")
else:
    print("Subdivision histogram returned None.")


## Step 4 — Differential Attenuation (MIU) Analysis

Compare Block 2 vs Block 4 (coal) against Block 1 and Block 3 (air references) using the pre-log FFC differential regression model.

In [ ]:
# ── Step 5: Full MIU analysis (differential regression) ───────────────────
comparison_result = compare_blocks_1_vs_3(image_file, subdivisions)

summary = comparison_result["summary"]
print("=== MIU Summary ===")
print(f"Orientation      : {summary['orientation']}")
print(f"μ Block 2 (coal) : {summary['mu_block2']:.5f} ± {summary['delta_mu_block2']:.5f}")
print(f"μ Block 4 (coal) : {summary['mu_block4']:.5f} ± {summary['delta_mu_block4']:.5f}")
print(f"R² Block 2       : {summary['r2_block2']:.4f}")
print(f"R² Block 4       : {summary['r2_block4']:.4f}")
if summary.get("air_validation_warning"):
    print(f"⚠️  {summary['air_validation_warning']}")
